# Week 8 — RNA-seq and Network Analysis

**From high-dimensional molecular measurements to systems-level structure**

This notebook contains two computational neuroscience case studies:

1. **RNA-seq:** identify genes whose expression differs between experimental groups.
2. **Network analysis:** represent interactions among brain regions as a graph and quantify network organization.

The two analyses ask different questions:

> **RNA-seq:** Which molecular features differ between conditions?

> **Network analysis:** How is a system organized, and does that organization differ between conditions?

## Learning goals

By the end of class you should be able to:

1. distinguish raw RNA-seq counts from normalized expression values;
2. interpret log2 fold change, p-values, and adjusted p-values;
3. read a volcano plot and expression heatmap;
4. explain why RNA-seq differential-expression tools use count-aware models;
5. represent connectivity data as an adjacency matrix;
6. define nodes, edges, degree/strength, density, and clustering;
7. identify hubs and compare network-level summaries between groups;
8. explain why thresholding and network construction choices affect conclusions.

The datasets are generated in this notebook so no external files are required.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests

import networkx as nx

rng = np.random.default_rng(42)

pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 3)


# Part I — RNA-seq

## 1. What does an RNA-seq count represent?

RNA-seq measures how many sequencing reads are assigned to each gene.

A simplified count matrix looks like this:

| Gene | Control 1 | Control 2 | Stimulated 1 | Stimulated 2 |
|---|---:|---:|---:|---:|
| Gene A | 120 | 138 | 350 | 372 |
| Gene B | 520 | 490 | 505 | 530 |
| Gene C | 18 | 25 | 7 | 11 |

The rows are genes and the columns are biological samples.

Raw counts are affected by both:

- the biological abundance of a transcript;
- the total number of reads sequenced from that sample.

Therefore, raw counts should not usually be compared directly across samples without normalization.


In [ ]:
# Generate a synthetic bulk RNA-seq experiment.

n_genes = 500
n_control = 6
n_stimulated = 6
n_samples = n_control + n_stimulated

genes = [f"Gene_{i:03d}" for i in range(1, n_genes + 1)]
sample_ids = [f"Control_{i+1}" for i in range(n_control)] + [
    f"Stimulated_{i+1}" for i in range(n_stimulated)
]

condition = np.array(
    ["Control"] * n_control + ["Stimulated"] * n_stimulated
)

# Baseline expression varies substantially across genes.
baseline_mean = np.exp(rng.normal(np.log(150), 1.0, n_genes))

# Simulate sequencing-depth differences across samples.
library_factor = rng.lognormal(mean=0, sigma=0.20, size=n_samples)

# A small subset of genes truly changes after stimulation.
true_log2fc = np.zeros(n_genes)

upregulated = np.arange(0, 18)
downregulated = np.arange(18, 30)

true_log2fc[upregulated] = rng.uniform(1.0, 2.2, len(upregulated))
true_log2fc[downregulated] = rng.uniform(-2.0, -1.0, len(downregulated))

counts = np.zeros((n_genes, n_samples), dtype=int)

# Gamma-Poisson / negative-binomial-like overdispersion.
dispersion = 0.15

for g in range(n_genes):
    for s in range(n_samples):
        fold_change = (
            2 ** true_log2fc[g]
            if condition[s] == "Stimulated"
            else 1.0
        )

        expected = baseline_mean[g] * library_factor[s] * fold_change

        shape = 1 / dispersion
        scale = expected * dispersion
        latent_rate = rng.gamma(shape=shape, scale=scale)

        counts[g, s] = rng.poisson(latent_rate)

count_df = pd.DataFrame(counts, index=genes, columns=sample_ids)

sample_metadata = pd.DataFrame({
    "sample_id": sample_ids,
    "condition": condition
})

print(count_df.shape)
sample_metadata


## 2. Library size and normalization

A sample with more total sequencing reads will tend to have larger counts for nearly every gene.

One simple normalization is **counts per million (CPM)**:

$$
CPM_{gs}
=
\frac{\text{count}_{gs}}
{\text{total reads}_s}
\times 10^6
$$

where:

- \(g\) indexes genes;
- \(s\) indexes samples.

We will then use

$$
\log_2(CPM + 1)
$$

for visualization.

This transformation compresses the large dynamic range of RNA-seq data.

**Important:** CPM and log transformation are useful teaching tools and visualization tools, but production differential-expression workflows commonly use count-aware methods such as DESeq2 or edgeR.


In [ ]:
library_sizes = count_df.sum(axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(library_sizes.index, library_sizes.values)
ax.set_ylabel("Total mapped counts")
ax.set_title("Library size varies across samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
cpm = count_df.div(count_df.sum(axis=0), axis=1) * 1_000_000
log_cpm = np.log2(cpm + 1)

log_cpm.iloc[:5, :6]


## 3. Effect size: log2 fold change

For each gene, compare mean expression between stimulated and control samples.

A common effect-size scale is the **log2 fold change**:

$$
\log_2 FC
=
\log_2
\left(
\frac{\text{stimulated expression}}
{\text{control expression}}
\right)
$$

Interpretation:

- log2FC = 0 → no change;
- log2FC = 1 → approximately 2-fold higher;
- log2FC = 2 → approximately 4-fold higher;
- log2FC = -1 → approximately 2-fold lower.

Effect size and statistical significance answer different questions.

A tiny effect can be statistically precise, while a large effect can be uncertain.


In [ ]:
control_cols = sample_metadata.loc[
    sample_metadata["condition"] == "Control",
    "sample_id"
]

stim_cols = sample_metadata.loc[
    sample_metadata["condition"] == "Stimulated",
    "sample_id"
]

mean_control = cpm[control_cols].mean(axis=1)
mean_stim = cpm[stim_cols].mean(axis=1)

log2fc = np.log2(
    (mean_stim + 1) /
    (mean_control + 1)
)

log2fc.head()


## 4. A simple differential-expression demonstration

For this classroom demonstration, we will perform a Welch t-test on log2(CPM + 1) values for each gene.

This is useful for learning how to interpret a differential-expression table.

It is **not** the preferred production method for RNA-seq because RNA-seq counts are discrete, mean-dependent, and overdispersed.

Count-aware methods such as DESeq2 and edgeR typically model the raw counts using negative-binomial distributions.


In [ ]:
p_values = []

for gene in genes:
    control_values = log_cpm.loc[gene, control_cols]
    stim_values = log_cpm.loc[gene, stim_cols]

    result = stats.ttest_ind(
        control_values,
        stim_values,
        equal_var=False
    )

    p_values.append(result.pvalue)

de_results = pd.DataFrame({
    "gene": genes,
    "mean_control_CPM": mean_control.values,
    "mean_stimulated_CPM": mean_stim.values,
    "log2FC": log2fc.values,
    "p_value": p_values
})

de_results["FDR"] = multipletests(
    de_results["p_value"],
    method="fdr_bh"
)[1]

de_results["significant"] = (
    (de_results["FDR"] < 0.05) &
    (np.abs(de_results["log2FC"]) >= 1)
)

de_results.sort_values("FDR").head(12)


### Interpretation prompt

For one row in the table, identify:

1. the gene;
2. the direction of the expression change;
3. the approximate fold change;
4. whether the statistical evidence remains strong after FDR correction.

A statistically significant gene is not automatically biologically important.

Questions to ask include:

- Is the effect large enough to matter?
- Is the gene expressed at a meaningful level?
- Is the result consistent across biological replicates?
- Does the finding make biological sense?
- Does it replicate in an independent experiment?


## 5. Volcano plot

A volcano plot combines effect size and statistical evidence.

Typical axes are:

- x-axis: log2 fold change;
- y-axis: \(-\log_{10}\) adjusted p-value.

Genes farther left or right have larger expression changes.

Genes higher on the plot have stronger statistical evidence.


In [ ]:
plot_df = de_results.copy()

plot_df["minus_log10_FDR"] = -np.log10(
    plot_df["FDR"].clip(lower=1e-300)
)

fig, ax = plt.subplots(figsize=(8, 6))

nonsig = ~plot_df["significant"]
sig = plot_df["significant"]

ax.scatter(
    plot_df.loc[nonsig, "log2FC"],
    plot_df.loc[nonsig, "minus_log10_FDR"],
    alpha=0.45,
    s=25,
    label="Not selected"
)

ax.scatter(
    plot_df.loc[sig, "log2FC"],
    plot_df.loc[sig, "minus_log10_FDR"],
    alpha=0.8,
    s=35,
    label="FDR < 0.05 and |log2FC| ≥ 1"
)

ax.axvline(-1, linestyle="--", linewidth=1)
ax.axvline(1, linestyle="--", linewidth=1)
ax.axhline(-np.log10(0.05), linestyle="--", linewidth=1)

ax.set_xlabel("log2 fold change")
ax.set_ylabel("-log10(FDR)")
ax.set_title("Differential-expression volcano plot")
ax.legend()
plt.show()


### Reading the plot

A point in the **upper right** represents a gene that is:

- higher in stimulated samples;
- strongly supported statistically.

A point in the **upper left** represents a gene that is:

- lower in stimulated samples;
- strongly supported statistically.

A point far to the right but near the bottom has a large estimated effect but weak statistical evidence.


## 6. Heatmap of the strongest expression differences

A volcano plot summarizes each gene with two numbers.

A heatmap lets us inspect the **sample-level pattern**.

This matters because a strong group average can sometimes be driven by one or two unusual samples.


In [ ]:
top_genes = (
    de_results
    .sort_values("FDR")
    .head(20)["gene"]
)

heat = log_cpm.loc[top_genes]

# Standardize each gene across samples so the heatmap emphasizes relative patterns.
heat_z = heat.sub(heat.mean(axis=1), axis=0).div(
    heat.std(axis=1).replace(0, np.nan),
    axis=0
)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(
    heat_z,
    aspect="auto",
    cmap="coolwarm",
    vmin=-2.5,
    vmax=2.5
)

ax.set_xticks(np.arange(len(sample_ids)))
ax.set_xticklabels(sample_ids, rotation=45, ha="right")

ax.set_yticks(np.arange(len(top_genes)))
ax.set_yticklabels(top_genes)

ax.set_xlabel("Sample")
ax.set_ylabel("Gene")
ax.set_title("Top differential-expression signals across samples")

plt.colorbar(im, ax=ax, label="Gene-wise z-score")
plt.tight_layout()
plt.show()


## 7. What RNA-seq can and cannot tell us

Differential expression can tell us that a gene's measured RNA abundance differs between groups.

It does not automatically establish:

- that the gene caused the phenotype;
- that protein abundance changed;
- that every cell type changed in the same way;
- that a pathway is activated;
- that the change occurred directly in response to treatment.

Bulk tissue is especially important to interpret carefully because measured expression can change when:

1. gene regulation changes within a cell type;
2. the proportions of cell types change;
3. both occur simultaneously.

RNA-seq is therefore a measurement layer that requires biological interpretation.


# Part II — Network Analysis

## 8. Represent a system as a graph

A network contains:

- **nodes**: the objects being studied;
- **edges**: relationships between those objects.

In neuroscience, nodes might be:

- neurons;
- brain regions;
- genes;
- proteins.

Edges might represent:

- anatomical connections;
- correlation;
- coherence;
- synaptic connections;
- gene co-expression.

Here, each node is a brain region and each edge is a simulated functional-connectivity strength.

The network can be stored in an **adjacency matrix**:

$$
A_{ij}
=
\text{connection strength between node } i \text{ and node } j.
$$

For an undirected network:

$$
A_{ij} = A_{ji}.
$$


In [ ]:
regions = [
    "Hippocampus",
    "PFC",
    "Amygdala",
    "Thalamus",
    "Striatum",
    "Motor",
    "Visual",
    "Somatosensory"
]

n_regions = len(regions)
n_network_animals = 12

network_conditions = np.array(
    ["Control"] * 6 + ["Stimulated"] * 6
)

# Baseline connectivity template.
base = np.array([
    [0.00, 0.52, 0.38, 0.34, 0.24, 0.18, 0.14, 0.16],
    [0.52, 0.00, 0.42, 0.36, 0.34, 0.24, 0.18, 0.20],
    [0.38, 0.42, 0.00, 0.39, 0.28, 0.18, 0.16, 0.17],
    [0.34, 0.36, 0.39, 0.00, 0.31, 0.25, 0.25, 0.27],
    [0.24, 0.34, 0.28, 0.31, 0.00, 0.44, 0.22, 0.26],
    [0.18, 0.24, 0.18, 0.25, 0.44, 0.00, 0.35, 0.48],
    [0.14, 0.18, 0.16, 0.25, 0.22, 0.35, 0.00, 0.46],
    [0.16, 0.20, 0.17, 0.27, 0.26, 0.48, 0.46, 0.00]
])

# Stimulation selectively strengthens several limbic/prefrontal connections.
stim_change = np.zeros_like(base)

for i, j, change in [
    (0, 1, 0.20),  # hippocampus-PFC
    (0, 2, 0.16),  # hippocampus-amygdala
    (1, 2, 0.14),  # PFC-amygdala
    (1, 3, 0.12),  # PFC-thalamus
]:
    stim_change[i, j] = change
    stim_change[j, i] = change

network_matrices = []

for condition_name in network_conditions:
    noise = rng.normal(0, 0.05, size=(n_regions, n_regions))
    noise = (noise + noise.T) / 2
    np.fill_diagonal(noise, 0)

    matrix = base + noise

    if condition_name == "Stimulated":
        matrix = matrix + stim_change

    matrix = np.clip(matrix, 0, 1)
    np.fill_diagonal(matrix, 0)

    network_matrices.append(matrix)

network_matrices = np.asarray(network_matrices)

network_matrices.shape


## 9. Inspect the adjacency matrix first

Before calculating graph metrics, look at the matrix.

A weighted adjacency matrix contains more information than a binary network because connection strength is preserved.

The diagonal is zero because a region is not being connected to itself in this example.


In [ ]:
control_mean_network = network_matrices[:6].mean(axis=0)
stim_mean_network = network_matrices[6:].mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, matrix, title in [
    (axes[0], control_mean_network, "Control mean connectivity"),
    (axes[1], stim_mean_network, "Stimulated mean connectivity")
]:
    im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=0.8)
    ax.set_xticks(np.arange(n_regions))
    ax.set_xticklabels(regions, rotation=45, ha="right")
    ax.set_yticks(np.arange(n_regions))
    ax.set_yticklabels(regions)
    ax.set_title(title)

fig.colorbar(im, ax=axes, label="Connectivity strength")
plt.tight_layout()
plt.show()


### Interpretation prompt

Before calculating any graph metric:

1. Which region pairs appear most strongly connected?
2. Which connections appear stronger in the stimulated group?
3. Does the matrix suggest one global network change or a more selective change?

Network metrics summarize structure, but the original matrix should remain visible during interpretation.


## 10. Convert the matrix into a graph

A graph gives a visual representation of the same adjacency matrix.

For visualization, we will only draw edges with weight at least 0.30.

This threshold is a **choice**.

Changing the threshold can change:

- the number of edges;
- which nodes appear to be hubs;
- network density;
- clustering;
- connected components.

Threshold selection should therefore be justified rather than chosen only because a graph looks cleaner.


In [ ]:
def graph_from_matrix(matrix, threshold=0.30):
    G = nx.Graph()

    for region in regions:
        G.add_node(region)

    for i in range(n_regions):
        for j in range(i + 1, n_regions):
            weight = matrix[i, j]

            if weight >= threshold:
                G.add_edge(
                    regions[i],
                    regions[j],
                    weight=float(weight)
                )

    return G

G_control = graph_from_matrix(control_mean_network)
G_stim = graph_from_matrix(stim_mean_network)

pos = nx.spring_layout(
    G_stim,
    seed=5,
    weight="weight"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, G, title in [
    (axes[0], G_control, "Control"),
    (axes[1], G_stim, "Stimulated")
]:
    weights = [3 * G[u][v]["weight"] for u, v in G.edges()]

    nx.draw_networkx(
        G,
        pos=pos,
        ax=ax,
        with_labels=True,
        node_size=1800,
        font_size=9,
        width=weights
    )

    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 11. Node degree and strength

For an unweighted network, **degree** is the number of edges connected to a node.

For a weighted network, **strength** is the sum of edge weights connected to that node:

$$
s_i
=
\sum_j A_{ij}.
$$

A node with high strength is strongly connected to many parts of the network.

Such nodes are sometimes described as **hubs**, but hub status depends on how the network was constructed.


In [ ]:
def node_strengths(matrix):
    return pd.Series(
        matrix.sum(axis=1),
        index=regions,
        name="strength"
    )

strength_table = pd.DataFrame({
    "Control": node_strengths(control_mean_network),
    "Stimulated": node_strengths(stim_mean_network)
})

strength_table["Change"] = (
    strength_table["Stimulated"] -
    strength_table["Control"]
)

strength_table.sort_values(
    "Stimulated",
    ascending=False
)


In [ ]:
plot_strength = strength_table.sort_values(
    "Stimulated"
)

fig, ax = plt.subplots(figsize=(8, 5))

y = np.arange(len(plot_strength))

ax.scatter(
    plot_strength["Control"],
    y,
    s=70,
    label="Control"
)

ax.scatter(
    plot_strength["Stimulated"],
    y,
    s=70,
    label="Stimulated"
)

for i, (_, row) in enumerate(plot_strength.iterrows()):
    ax.plot(
        [row["Control"], row["Stimulated"]],
        [i, i],
        linewidth=1
    )

ax.set_yticks(y)
ax.set_yticklabels(plot_strength.index)
ax.set_xlabel("Node strength")
ax.set_title("Connectivity strength by brain region")
ax.legend()
plt.show()


## 12. Network-level summaries

Several graph metrics summarize different aspects of network organization.

### Density

For a binary undirected network:

$$
Density
=
\frac{\text{number of observed edges}}
{\text{number of possible edges}}.
$$

A denser network contains a larger fraction of possible connections.

### Clustering coefficient

The clustering coefficient asks whether a node's neighbors also tend to connect to each other.

High clustering can indicate locally interconnected groups.

### Average node strength

For a weighted network, average strength summarizes overall connection magnitude.

No single network metric captures the entire system.


In [ ]:
def network_metrics(matrix, threshold=0.30):
    G = graph_from_matrix(matrix, threshold=threshold)

    strengths = [
        strength
        for _, strength in G.degree(weight="weight")
    ]

    return {
        "density": nx.density(G),
        "mean_clustering": nx.average_clustering(G),
        "mean_strength": np.mean(strengths)
    }

animal_network_metrics = []

for i, matrix in enumerate(network_matrices):
    metrics = network_metrics(matrix)

    metrics["animal"] = f"A{i+1:02d}"
    metrics["condition"] = network_conditions[i]

    animal_network_metrics.append(metrics)

network_metrics_df = pd.DataFrame(animal_network_metrics)

network_metrics_df


In [ ]:
metrics_to_plot = [
    "density",
    "mean_clustering",
    "mean_strength"
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13, 4)
)

for ax, metric in zip(axes, metrics_to_plot):

    for condition_name, group in network_metrics_df.groupby("condition"):
        x = 0 if condition_name == "Control" else 1
        jitter = np.linspace(
            -0.06,
            0.06,
            len(group)
        )

        ax.scatter(
            np.full(len(group), x) + jitter,
            group[metric],
            s=60
        )

    ax.set_xticks(
        [0, 1],
        ["Control", "Stimulated"]
    )

    ax.set_title(metric.replace("_", " ").title())

plt.tight_layout()
plt.show()


## 13. Threshold sensitivity

A network metric can change when the threshold changes.

This does not necessarily mean the metric is useless.

It means the analysis depends on a modeling choice.

One useful sensitivity analysis is to calculate the same network summary across several thresholds.


In [ ]:
thresholds = np.arange(
    0.20,
    0.51,
    0.05
)

rows = []

for threshold in thresholds:

    for condition_name, matrix in [
        ("Control", control_mean_network),
        ("Stimulated", stim_mean_network)
    ]:
        G = graph_from_matrix(
            matrix,
            threshold=threshold
        )

        rows.append({
            "threshold": threshold,
            "condition": condition_name,
            "density": nx.density(G),
            "clustering": nx.average_clustering(G)
        })

threshold_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4.5))

for condition_name, group in threshold_df.groupby("condition"):
    ax.plot(
        group["threshold"],
        group["density"],
        marker="o",
        label=condition_name
    )

ax.set_xlabel("Connectivity threshold")
ax.set_ylabel("Network density")
ax.set_title("Network conclusions depend on construction choices")
ax.legend()
plt.show()


## 14. What does a connectivity edge mean?

A network edge is only meaningful in the context of how it was defined.

An edge could represent:

- anatomical connectivity;
- correlation;
- coherence;
- covariance;
- mutual information;
- effective connectivity;
- a tractography estimate.

These quantities are not interchangeable.

A strong functional-connectivity edge does not automatically mean:

- the regions are directly anatomically connected;
- one region causes activity in the other;
- information flows in a particular direction.

The mathematical network inherits the assumptions and limitations of the measurement used to construct it.


# Final synthesis

RNA-seq and network analysis reduce complex datasets in different ways.

## RNA-seq

Start with:

$$
\text{genes} \times \text{samples}
$$

and ask which molecular features differ between conditions.

Important outputs include:

- normalized expression;
- log2 fold change;
- adjusted statistical evidence;
- sample-level expression patterns.

## Network analysis

Start with:

$$
\text{nodes} \times \text{nodes}
$$

and ask how relationships are organized.

Important outputs include:

- adjacency matrices;
- node strength or degree;
- hubs;
- density;
- clustering;
- sensitivity to network-construction choices.

## Final questions

1. Why should raw RNA-seq counts not usually be compared directly across samples?
2. What does a log2 fold change of +1 mean?
3. Why can a volcano plot be misleading if you never inspect sample-level expression?
4. What are the nodes and edges in the connectivity network?
5. How is node strength different from degree?
6. Why can threshold selection change network conclusions?
7. Why does a connectivity edge not automatically imply direct anatomical or causal connectivity?
8. In both analyses, where do analyst choices enter before the final figure is created?

## Key idea

Complex biological data become interpretable only after we choose a representation.

Those choices—normalization, statistical modeling, edge definition, thresholding, and summary metrics—are part of the scientific analysis, not merely technical details.
